In [ ]:
from pulao.logging import init_logging
import logging
from IPython.display import display

from pulao.constant import FractalType
from pulao.trend import TrendManager
from pulao.swing import SwingManager
from pulao.bar import SBar, SBarManager, CBarManager, CBar
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import BarData

import polars as pl

init_logging(level=logging.DEBUG)

df = pl.read_csv("../dataset/I8888.XDCE_60m.csv", try_parse_dates=True)
df = df.head(100)  # test

sbar_list = []
columns = df.columns

for idx, row in enumerate(df.iter_rows()):
    row_dict = dict(zip(columns, row))
    # datetime,open,close,high,low,volume,money,open_interest,signal
    datetime = row_dict["datetime"]
    open = row_dict["open"]
    close = row_dict["close"]
    high = row_dict["high"]
    low = row_dict["low"]
    volume = row_dict["volume"]
    money = row_dict["money"]
    open_interest = row_dict["open_interest"]

    bar = BarData(
        gateway_name="ctp-test",
        symbol="i8888",
        exchange=Exchange.SHFE,
        interval=Interval.MINUTE,
        datetime=datetime,
        open_price=open,
        close_price=close,
        high_price=high,
        low_price=low,
        volume=volume,
        open_interest=open_interest,
        turnover=money,
    )
    sbar = SBar(index= len(sbar_list), symbol=bar.symbol, exchange=bar.exchange.value, timeframe=bar.interval.value, datetime=datetime, turnover=money, open_price=open, close_price=close, high_price=high, low_price=low, volume=volume)

    sbar_list.append(sbar)
# 模拟行情数据接收
sbar_manager = SBarManager()
cbar_manager = CBarManager(sbar_manager=sbar_manager)
swing_manager = SwingManager(cbar_manager=cbar_manager)
# trend_manager = TrendManager(swing_manager=swing_manager)

# 流入模拟数据
for sbar in sbar_list:
    sbar_manager.append(sbar)

#
display(swing_manager.df_swing)

cbar_manager.df_swing

# swing_manager.df_swing

In [ ]:
from pulao.constant import Direction

# display(df_swing)
display(df_trend)

# df_swing["start_datetime"] = pd.Series(dtype="datetime64[ns]")
# df_swing["end_datetime"] = pd.Series(dtype="datetime64[ns]")
#
# for i, row in enumerate(df_swing.itertuples()):
#     df = df_swing[(df_swing["index"]>=row.start_index) & (df_swing["index"]<=row.end_index)]
#     start_datetime = df.iloc[0]["datetime"]
#     end_datetime = df.iloc[-1]["datetime"]
#     df_swing.at[i, "start_datetime"] = start_datetime
#     df_swing.at[i, "end_datetime"] = end_datetime
#
# xs=[]
# ys=[]
# for i, row in enumerate(df_swing.itertuples()):
#     xs += [row.start_datetime, row.end_datetime, None]
#     if row.direction == SwingDirection.UP:
#         start_price = df_swing.at[row.start_index,"low_price"]
#         end_price = df_swing.at[row.end_index,"high_price"]
#     else:
#         start_price = df_swing.at[row.start_index,"high_price"]
#         end_price = df_swing.at[row.end_index,"low_price"]
#     ys += [start_price, end_price, None]
# display(xs,ys)
